# 🎬 AI คนตัดต่อ ของ Live AI (Qwen3.5-9B บนการ์ดจอฟรีของ Colab)

1. เมนู **รันไทม์ → เปลี่ยนประเภทรันไทม์ → T4 GPU** → บันทึก
2. วาง **รหัสเชื่อม** ที่คัดลอกจากสตูดิโอ (แท็บ ตั้งค่า & ส่งออก → สร้างรหัสเชื่อม) ในช่องด้านล่าง
3. กด ▶ ที่ช่องโค้ด แล้วรอจนขึ้นว่า **เชื่อมต่อแล้ว** (ครั้งแรกประมาณ 5–10 นาที)
4. กลับไปกด **Start** ในสตูดิโอได้เลย — ปิดจอมือถือได้ แต่อย่าปิดแท็บนี้

AI ทำงานในบัญชี Google ของคุณ ภาพจากคลิปส่งตรงระหว่าง Live AI ของคุณกับ Colab นี้เท่านั้น
โมเดล: Qwen3.5-9B (Apache 2.0) จาก Unsloth GGUF · โปรแกรม: llama.cpp

In [ ]:
#@title ▶ เปิด AI คนตัดต่อ { display-mode: "form" }
รหัสเชื่อม = ""  #@param {type:"string"}
import os, subprocess, sys, time, json, glob, urllib.request, tarfile, shutil
REPO = "mamaff9522-sketch/live-ai"
code = รหัสเชื่อม.strip()
if "#lew_" not in code:
    raise SystemExit("❌ ยังไม่ได้วางรหัสเชื่อม หรือรหัสไม่ถูกต้อง: คัดลอกจากสตูดิโอ แท็บ ตั้งค่า & ส่งออก → สร้างรหัสเชื่อม")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("❌ ยังไม่ได้เปิดการ์ดจอ: เมนู รันไทม์ → เปลี่ยนประเภทรันไทม์ → T4 GPU แล้วกด ▶ อีกครั้ง")
print("✅ การ์ดจอ:", gpu)

# 1) llama-server ที่สร้างไว้แล้วสำหรับการ์ดจอ Colab (ถ้าไม่มี จะคอมไพล์เอง ~20 นาที)
os.makedirs("/content/llama", exist_ok=True)
if not os.path.exists("/content/llama/llama-server"):
    try:
        url = f"https://github.com/{REPO}/releases/download/tools/llama-server-cuda.tar.gz"
        urllib.request.urlretrieve(url, "/content/llama.tgz")
        tarfile.open("/content/llama.tgz").extractall("/content/llama")
        os.chmod("/content/llama/llama-server", 0o755)
        print("✅ ดาวน์โหลด llama-server แล้ว")
    except Exception as e:
        print("⚠️ ดาวน์โหลดไม่ได้ (", e, ") — คอมไพล์เอง ประมาณ 20 นาที")
        subprocess.run("git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/src && "
                       "cmake -S /content/src -B /content/src/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF "
                       "-DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF > /dev/null && "
                       "cmake --build /content/src/build --target llama-server -j4 > /dev/null && "
                       "cp -a /content/src/build/bin/* /content/llama/", shell=True, check=True)
env = dict(os.environ, LD_LIBRARY_PATH="/content/llama:" + os.environ.get("LD_LIBRARY_PATH", ""))

# 2) โมเดล Qwen3.5-9B (Q4_K_M) + ส่วนดูภาพ (mmproj)
from huggingface_hub import HfApi, hf_hub_download
MODEL_REPO = "unsloth/Qwen3.5-9B-GGUF"
files = [f.rfilename for f in HfApi().model_info(MODEL_REPO, files_metadata=False).siblings]
gguf = next(f for f in files if f.endswith(".gguf") and "Q4_K_M" in f and "mmproj" not in f.lower())
mm = sorted([f for f in files if "mmproj" in f.lower()], key=lambda f: (0 if "F16" in f.upper() else 1 if "BF16" in f.upper() else 2))[0]
print("⬇️ ดาวน์โหลด", gguf, "และ", mm, "(ครั้งแรกประมาณ 5–7 GB)")
mp = hf_hub_download(MODEL_REPO, gguf, local_dir="/content/models")
pp = hf_hub_download(MODEL_REPO, mm, local_dir="/content/models")

# 3) เปิดโมเดลบนการ์ดจอ
subprocess.run("pkill -f llama-server", shell=True)
log = open("/content/llama-server.log", "w")
srv = subprocess.Popen(["/content/llama/llama-server", "-m", mp, "--mmproj", pp, "-ngl", "99", "-c", "16384", "-np", "1",
                        "--jinja", "--host", "127.0.0.1", "--port", "8080", "--no-webui"], stdout=log, stderr=log, env=env)
for i in range(240):
    try:
        if json.load(urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=3)).get("status") == "ok":
            break
    except Exception:
        pass
    if srv.poll() is not None:
        print(open("/content/llama-server.log").read()[-2000:])
        raise SystemExit("❌ เปิดโมเดลไม่สำเร็จ (ดูข้อความด้านบน)")
    time.sleep(2)
print("✅ โมเดลพร้อมบนการ์ดจอ")

# 4) เชื่อมกับ Live AI แล้วรอรับงานตัดต่อ (ปล่อยช่องนี้ทำงานไว้)
urllib.request.urlretrieve(f"https://raw.githubusercontent.com/{REPO}/main/scripts/ai_editor_worker.py", "/content/ai_editor_worker.py")
print("🔗 กำลังเชื่อมกับ Live AI …")
subprocess.run([sys.executable, "-u", "/content/ai_editor_worker.py", "--code", code, "--llm", "http://127.0.0.1:8080"])
